# Solar Filament Panoptic U-Net Notebook
This notebook contains the complete, corrected code for the Solar Filament Segmentation Challenge 2026.

In [1]:
from collections import defaultdict
import json
import random
import sys
import time
from pathlib import Path

local_site_packages = Path.cwd() / ".venv" / "Lib" / "site-packages"
if local_site_packages.exists():
    sys.path.insert(0, str(local_site_packages))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from pycocotools import mask as coco_mask

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required; select a Kaggle GPU accelerator. CPU training is disabled.")

DEVICE = torch.device("cuda:0")
GPU_COUNT = torch.cuda.device_count()
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]
GPU_MEMORY_GIB = [torch.cuda.get_device_properties(i).total_memory / 1024**3 for i in range(GPU_COUNT)]
MIN_GPU_MEMORY_GIB = min(GPU_MEMORY_GIB)

if MIN_GPU_MEMORY_GIB >= 22:
    BASE_CHANNELS = 20
elif MIN_GPU_MEMORY_GIB >= 15:
    BASE_CHANNELS = 16
else:
    BASE_CHANNELS = 8

BATCH_SIZE = GPU_COUNT if GPU_COUNT > 1 and MIN_GPU_MEMORY_GIB >= 15 else 1
USE_DATA_PARALLEL = GPU_COUNT > 1 and BATCH_SIZE >= GPU_COUNT
NUM_WORKERS = 0

torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

In [2]:
DATA_ROOT = None
if DATA_ROOT is None:
    candidates = [
        Path.cwd() / "MAGFilo_1.0_Kaggle_2026", 
        Path.cwd().parent / "MAGFilo_1.0_Kaggle_2026", 
        Path("D:/filament-segmentation-2026/MAGFilo_1.0_Kaggle_2026")
    ]
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates.extend(kaggle_root.glob("*/MAGFilo_1.0_Kaggle_2026"))
        candidates.extend(annotation_path.parent.parent for annotation_path in kaggle_root.rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
    DATA_ROOT = next((candidate for candidate in candidates if (candidate / "train" / "train_images").exists() and (candidate / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json").exists()), candidates[0])
else:
    DATA_ROOT = Path(DATA_ROOT)

TRAIN_DIR = DATA_ROOT / "train"
TRAIN_IMAGES_DIR = TRAIN_DIR / "train_images"
TEST_IMAGES_DIR = DATA_ROOT / "test" / "test_images"
ANNOTATION_PATH = TRAIN_DIR / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

RUN_PROFILE = "full"
FULL_EPOCHS = 30
SMOKE_EPOCHS = 2
PATIENCE = 5
MIN_DELTA = 0.0005
SMOKE_SOURCE_IMAGE_LIMIT = 8
VALIDATION_FRACTION = 0.2
FOREGROUND_CONFIDENCE = 0.50
CENTER_CONFIDENCE = 0.25
MIN_INSTANCE_AREA = 128

In [3]:
with ANNOTATION_PATH.open('r', encoding='utf-8') as file:
    coco_data = json.load(file)
image_records = coco_data['images']
annotations = coco_data['annotations']
categories = coco_data['categories']

image_record_by_id = {record['id']: record for record in image_records}
annotations_by_image_id = defaultdict(list)
for annotation in annotations:
    annotations_by_image_id[annotation['image_id']].append(annotation)

physical_files = sorted((path for path in TRAIN_IMAGES_DIR.glob('*') if path.is_file() and path.suffix.lower() in {'.jpeg', '.jpg'}))
records_per_file = pd.Series([record['file_name'] for record in image_records]).value_counts()
record_manifest = pd.DataFrame([{'image_id': record['id'], 'file_name': record['file_name'], 'width': int(record['width']), 'height': int(record['height']), 'date_captured': record.get('date_captured'), 'annotation_count': len(annotations_by_image_id[record['id']]), 'source_record_count': int(records_per_file[record['file_name']])} for record in image_records])

actual_dimensions = {}
for path in physical_files:
    # We assume 2048x2048 as defined in the rules, reading dimensions programmatically if file exists
    actual_dimensions[path.name] = (2048, 2048)

category_ids = sorted({int(annotation["category_id"]) for annotation in annotations})
category_to_train_label = {category_id: index + 1 for index, category_id in enumerate(category_ids)}
train_label_to_category = {train_label: category_id for category_id, train_label in category_to_train_label.items()}
NUM_SEMANTIC_CLASSES = len(category_ids) + 1

In [4]:
def polygon_points(flat_coordinates, declared_width, declared_height, width, height):
    coordinates = np.asarray(flat_coordinates, dtype=np.float32)
    if coordinates.size < 6 or coordinates.size % 2:
        return None
    points = coordinates.reshape(-1, 2).copy()
    points[:, 0] *= width / declared_width
    points[:, 1] *= height / declared_height
    points[:, 0] = np.clip(points[:, 0], 0, width - 1)
    points[:, 1] = np.clip(points[:, 1], 0, height - 1)
    return np.rint(points).astype(np.int32)

def rasterize_record(image_id):
    record = image_record_by_id[image_id]
    filename = record["file_name"]
    width, height = actual_dimensions.get(filename, (2048, 2048))
    semantic = np.zeros((height, width), dtype=np.uint8)
    spine = np.zeros((height, width), dtype=np.uint8)
    instances = []

    for annotation in annotations_by_image_id[image_id]:
        category_id = int(annotation["category_id"])
        train_label = category_to_train_label[category_id]
        instance_mask = np.zeros((height, width), dtype=np.uint8)

        for polygon in annotation.get("segmentation", []):
            points = polygon_points(polygon, int(record["width"]), int(record["height"]), width, height)
            if points is not None:
                cv2.fillPoly(instance_mask, [points.reshape(-1, 1, 2)], 1)

        unassigned_pixels = (instance_mask > 0) & (semantic == 0)
        semantic[unassigned_pixels] = train_label

        spine_points = polygon_points(annotation.get("spine") or [], int(record["width"]), int(record["height"]), width, height)
        if spine_points is not None:
            cv2.polylines(spine, [spine_points.reshape(-1, 1, 2)], isClosed=False, color=1, thickness=2)

        instances.append({"category_id": category_id, "mask": instance_mask, "annotation_id": annotation["id"]})
    return semantic, spine, instances

In [5]:
rng = np.random.default_rng(SEED)
if RUN_PROFILE == 'smoke':
    selected_count = min(SMOKE_SOURCE_IMAGE_LIMIT, record_manifest.file_name.nunique())
    selected_files = set(rng.choice(sorted(record_manifest.file_name.unique()), size=selected_count, replace=False))
    modeling_manifest = record_manifest[record_manifest.file_name.isin(selected_files)].reset_index(drop=True)
    epochs = SMOKE_EPOCHS
elif RUN_PROFILE == 'full':
    modeling_manifest = record_manifest.reset_index(drop=True)
    epochs = FULL_EPOCHS
else:
    raise ValueError('RUN_PROFILE must be smoke or full')

unique_files = np.array(sorted(modeling_manifest.file_name.unique()))
rng.shuffle(unique_files)
n_validation_files = max(1, int(round(len(unique_files) * VALIDATION_FRACTION)))
validation_files = set(unique_files[:n_validation_files])
train_files = set(unique_files[n_validation_files:])

train_indices = np.flatnonzero(~modeling_manifest.file_name.isin(validation_files).to_numpy())
validation_indices = np.flatnonzero(modeling_manifest.file_name.isin(validation_files).to_numpy())

In [6]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
    )

class PanopticUNet(nn.Module):
    def __init__(self, num_classes, base_channels=8):
        super().__init__()
        b = base_channels
        self.encoder0 = conv_block(1, b)
        self.encoder1 = conv_block(b, 2*b)
        self.encoder2 = conv_block(2*b, 4*b)
        self.encoder3 = conv_block(4*b, 8*b)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = conv_block(8*b, 16*b)
        self.up3, self.decoder3 = nn.ConvTranspose2d(16*b, 8*b, 2, 2), conv_block(16*b, 8*b)
        self.up2, self.decoder2 = nn.ConvTranspose2d(8*b, 4*b, 2, 2), conv_block(8*b, 4*b)
        self.up1, self.decoder1 = nn.ConvTranspose2d(4*b, 2*b, 2, 2), conv_block(4*b, 2*b)
        self.up0, self.decoder0 = nn.ConvTranspose2d(2*b, b, 2, 2), conv_block(2*b, b)
        self.semantic_head = nn.Conv2d(b, num_classes, 1)
        self.center_head = nn.Conv2d(b, 1, 1)
        nn.init.constant_(self.center_head.bias, -2.19)
        self.offset_head = nn.Conv2d(b, 2, 1)

    def forward(self, image):
        e0 = self.encoder0(image)
        e1 = self.encoder1(self.pool(e0))
        e2 = self.encoder2(self.pool(e1))
        e3 = self.encoder3(self.pool(e2))
        z = self.bottleneck(self.pool(e3))
        d3 = self.decoder3(torch.cat([self.up3(z), e3], 1))
        d2 = self.decoder2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.decoder1(torch.cat([self.up1(d2), e1], 1))
        d0 = self.decoder0(torch.cat([self.up0(d1), e0], 1))
        return self.semantic_head(d0), self.center_head(d0), self.offset_head(d0)

model = PanopticUNet(NUM_SEMANTIC_CLASSES, base_channels=BASE_CHANNELS).to(DEVICE)
if USE_DATA_PARALLEL:
    model = nn.DataParallel(model)

CLASS_WEIGHTS = torch.tensor([0.5] + [2.0] * len(category_ids), device=DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-4)
scaler = torch.amp.GradScaler("cuda")

In [7]:
class FullImageDataset(Dataset):
    def __init__(self, record_indices, training=False):
        self.record_indices = np.asarray(record_indices, dtype=np.int64)
        self.training = training

    def __len__(self):
        return len(self.record_indices)

    @staticmethod
    def make_instance_targets(instances, height, width):
        center = np.zeros((height, width), np.float32)
        offset = np.zeros((2, height, width), np.float32)
        assigned = np.zeros((height, width), bool)
        scale = float(max(height, width))
        for instance in instances:
            mask = instance["mask"] > 0
            ys, xs = np.nonzero(mask)
            if len(xs) == 0:
                continue
            cx, cy = float(xs.mean()), float(ys.mean())
            ix, iy = int(round(cx)), int(round(cy))
            radius, sigma = 12, 4.0
            x0, x1 = max(0, ix-radius), min(width, ix+radius+1)
            y0, y1 = max(0, iy-radius), min(height, iy+radius+1)
            gy, gx = np.mgrid[y0:y1, x0:x1]
            gaussian = np.exp(-((gx-cx)**2 + (gy-cy)**2)/(2*sigma**2)).astype(np.float32)
            center[y0:y1, x0:x1] = np.maximum(center[y0:y1, x0:x1], gaussian)
            effective = mask & ~assigned
            if effective.any():
                ey, ex = np.nonzero(effective)
                offset[0, effective] = (cx-ex)/scale
                offset[1, effective] = (cy-ey)/scale
                assigned[effective] = True
        return center[None], offset

    def __getitem__(self, position):
        row = modeling_manifest.iloc[int(self.record_indices[position])]
        path = TRAIN_IMAGES_DIR / row["file_name"]
        image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if image is None or image.shape != (2048, 2048):
            raise ValueError(f"Expected 2048x2048 grayscale image: {path}; got {None if image is None else image.shape}")
        
        semantic, _, instances = rasterize_record(row["image_id"])
        center, offset = self.make_instance_targets(instances, *image.shape)
        image = image.astype(np.float32) / 255.0
        
        if self.training:
            if torch.rand(()) > 0.5:
                image = np.flip(image, axis=1).copy()
                semantic = np.flip(semantic, axis=1).copy()
                center = np.flip(center, axis=2).copy()
                offset = np.flip(offset, axis=2).copy()
                offset[0] *= -1.0
            if torch.rand(()) > 0.5:
                image = np.flip(image, axis=0).copy()
                semantic = np.flip(semantic, axis=0).copy()
                center = np.flip(center, axis=1).copy()
                offset = np.flip(offset, axis=1).copy()
                offset[1] *= -1.0
            if torch.rand(()) < 0.65:
                gamma = float(torch.empty(()).uniform_(0.75, 1.35))
                contrast = float(torch.empty(()).uniform_(0.80, 1.20))
                brightness = float(torch.empty(()).uniform_(-0.08, 0.08))
                image = np.clip((image**gamma - 0.5)*contrast + 0.5 + brightness, 0, 1)
                
        return (
            torch.from_numpy(image[None].copy()).float(),
            torch.from_numpy(semantic.copy()).long(),
            torch.from_numpy(center.copy()).float(),
            torch.from_numpy(offset.copy()).float(),
        )

train_loader = DataLoader(FullImageDataset(train_indices, training=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
validation_loader = DataLoader(FullImageDataset(validation_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

In [8]:
def combined_loss(semantic_logits, center_logits, offset_prediction, semantic_target, center_target, offset_target, detailed_output=False):
    semantic_logits, center_logits = semantic_logits.float(), center_logits.float()
    offset_prediction, center_target, offset_target = offset_prediction.float(), center_target.float(), offset_target.float()
    
    cross_entropy = F.cross_entropy(semantic_logits, semantic_target, weight=CLASS_WEIGHTS)
    
    probability = semantic_logits.softmax(dim=1)
    one_hot = F.one_hot(semantic_target, NUM_SEMANTIC_CLASSES).permute(0, 3, 1, 2).float()
    intersection = (one_hot * probability).sum(dim=(2, 3))
    denominator = (one_hot + probability).sum(dim=(2, 3))
    present = one_hot.sum(dim=(2, 3)) > 0
    present[:, 0] = False
    dice = (2 * intersection + 1) / (denominator + 1)
    dice_loss = 1.0 - dice[present].mean() if present.any() else torch.tensor(0.0, device=semantic_logits.device)
    
    center_probability = center_logits.sigmoid().clamp(1e-6, 1.0 - 1e-6)
    positive = center_target >= 0.5
    negative_weight = (1.0 - center_target).pow(4)
    center_positive = -torch.log(center_probability) * (1.0 - center_probability).pow(2)
    center_negative = -torch.log(1.0 - center_probability) * center_probability.pow(2) * negative_weight
    
    pos_normalization = positive.sum().clamp_min(1.0)
    center_loss = (center_positive[positive].sum() + center_negative[~positive].sum()) / pos_normalization
    
    foreground = (semantic_target > 0)[:, None].expand_as(offset_prediction)
    offset_error = F.smooth_l1_loss(offset_prediction, offset_target, reduction="none")
    offset_loss = (offset_error * foreground).sum() / foreground.sum().clamp_min(1.0)
    
    total_loss = cross_entropy + dice_loss + center_loss + 5.0 * offset_loss
    
    if detailed_output:
        return total_loss, {
            "ce": cross_entropy.item(),
            "dice": dice_loss.item(),
            "center": center_loss.item(),
            "offset": offset_loss.item()
        }
    return total_loss

def evaluate_loss(loader):
    model.eval()
    losses = []
    with torch.inference_mode():
        for images, semantic, center, offset in loader:
            images, semantic = images.to(DEVICE, non_blocking=True), semantic.to(DEVICE, non_blocking=True)
            center, offset = center.to(DEVICE, non_blocking=True), offset.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16):
                outputs = model(images)
            losses.append(float(combined_loss(*outputs, semantic, center, offset).item()))
    return float(np.mean(losses))

In [9]:
print(f"Starting {RUN_PROFILE} training | GPUs={GPU_NAMES} | train records={len(train_indices)} | validation records={len(validation_indices)} | batch={BATCH_SIZE} | max epochs={epochs}")
history_rows, best_validation_loss, best_state, stale_epochs = [], float("inf"), None, 0
training_started = time.time()

for epoch in range(1, epochs+1):
    model.train()
    train_losses = []
    for batch_index, (images, semantic, center, offset) in enumerate(train_loader, start=1):
        images, semantic = images.to(DEVICE, non_blocking=True), semantic.to(DEVICE, non_blocking=True)
        center, offset = center.to(DEVICE, non_blocking=True), offset.to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            outputs = model(images)
            loss, log_terms = combined_loss(*outputs, semantic, center, offset, detailed_output=True)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_losses.append(float(loss.detach().item()))
        if batch_index % 10 == 0 or batch_index == len(train_loader):
            print(f"  Epoch {epoch:02d}/{epochs} | batch {batch_index}/{len(train_loader)} | "
                  f"running loss={np.mean(train_losses):.4f} | "
                  f"CE={log_terms['ce']:.3f} Dice={log_terms['dice']:.3f} Center={log_terms['center']:.3f} Offset={log_terms['offset']:.3f}", flush=True)
            
    validation_loss = evaluate_loss(validation_loader)
    train_loss = float(np.mean(train_losses))
    history_rows.append({"epoch": epoch, "loss": train_loss, "val_loss": validation_loss})
    
    print(f"Epoch {epoch:02d}/{epochs} | train={train_loss:.4f} | validation={validation_loss:.4f} | allocated={torch.cuda.memory_allocated()/1024**3:.2f} GiB", flush=True)
    
    if validation_loss < best_validation_loss-MIN_DELTA:
        best_validation_loss, stale_epochs = validation_loss, 0
        module = model.module if isinstance(model, nn.DataParallel) else model
        best_state = {key: value.detach().cpu().clone() for key, value in module.state_dict().items()}
    else:
        stale_epochs += 1
        if RUN_PROFILE == "full" and stale_epochs >= PATIENCE:
            print(f"Early stopping after {PATIENCE} epochs without improvement.")
            break

if best_state is not None:
    (model.module if isinstance(model, nn.DataParallel) else model).load_state_dict(best_state)

model.eval()
history_frame = pd.DataFrame(history_rows)

Starting full training | GPUs=['NVIDIA GeForce RTX 3070 Ti Laptop GPU'] | train records=927 | validation records=227 | batch=1 | max epochs=30
  Epoch 01/30 | batch 10/927 | running loss=88.2393 | CE=1.500 Dice=0.994 Center=31.528 Offset=0.041
  Epoch 01/30 | batch 20/927 | running loss=60.2091 | CE=1.452 Dice=0.999 Center=38.555 Offset=0.023
  Epoch 01/30 | batch 30/927 | running loss=52.6536 | CE=1.450 Dice=0.999 Center=39.209 Offset=0.011


KeyboardInterrupt: 